# Level 3: Advanced Agent Capabilities with Prompt Chaining and ReAct Agent

Building on the simple agent introduced in [Level 2](Level2_simple_agent_with_websearch.ipynb), this tutorial continues the agent-focused section of our series by introducing techniques that make the agent smarter and more autonomous: **Prompt Chaining** and the **ReAct (Reasoning + Acting) framework**. These approaches allow the agent to complete multi-step tasks, dynamically choose tools, and adjust its behavior based on context.

- **Prompt Chaining** connects multiple prompts into a coherent sequence, allowing the agent to maintain context and perform multi-step reasoning across tool invocations. 
- **ReAct Agent** combines reasoning and acting steps in a loop, enabling the agent to make decisions, use tools dynamically, and adapt based on intermediate results. 

## Overview

In this notebook, you'll explore three agent configurations:
1. **Simple Agent (Baseline)** – Uses a single web search tool.
2. **Prompt Chaining** – Performs structured, multi-step reasoning by chaining prompts and responses.
3. **ReAct Agent** – Dynamically plans and executes actions using a loop of reasoning and tool use.


## Prerequisites

Before starting this notebook, ensure that you have:
- Followed the instructions in the [Setup Guide](./Level0_getting_started_with_Llama_Stack.ipynb) notebook. 
- A Tavily API key. It is critical for this notebook to run correctly. You can register for one at [https://tavily.com/](https://tavily.com/).

## 1. Setting Up this Notebook
We will start with a few imports needed for this demo only.

In [1]:
from llama_stack_client import Agent
from llama_stack_client.lib.agents.event_logger import EventLogger
from llama_stack_client.lib.agents.react.agent import ReActAgent
from llama_stack_client.lib.agents.react.tool_parser import ReActOutput
import sys
sys.path.append('..') 
from src.client_tools import get_location

Next, we will initialize our environment as described in detail in our ["Getting Started" notebook](./Level0_getting_started_with_Llama_Stack.ipynb). Please refer to it for additional explanations.

In [2]:
# for accessing the environment variables
import os
from dotenv import load_dotenv
load_dotenv()

# for communication with Llama Stack
from llama_stack_client import LlamaStackClient

# pretty print of the results returned from the model/agent
import sys
sys.path.append('..')  
from src.utils import step_printer
from termcolor import cprint

base_url = os.getenv("REMOTE_BASE_URL")


# Tavily search API key is required for some of our demos and must be provided to the client upon initialization.
# We will cover it in the agentic demos that use the respective tool. Please ignore this parameter for all other demos.
tavily_search_api_key = os.getenv("TAVILY_SEARCH_API_KEY")
if tavily_search_api_key is None:
    provider_data = None
else:
    provider_data = {"tavily_search_api_key": tavily_search_api_key}


client = LlamaStackClient(
    base_url=base_url,
    provider_data=provider_data
)
    
print(f"Connected to Llama Stack server")

# model_id for the model you wish to use that is configured with the Llama Stack server
model_id = "granite32-8b"

temperature = float(os.getenv("TEMPERATURE", 0.0))
if temperature > 0.0:
    top_p = float(os.getenv("TOP_P", 0.95))
    strategy = {"type": "top_p", "temperature": temperature, "top_p": top_p}
else:
    strategy = {"type": "greedy"}

max_tokens = int(os.getenv("MAX_TOKENS", 4096))

# sampling_params will later be used to pass the parameters to Llama Stack Agents/Inference APIs
sampling_params = {
    "strategy": strategy,
    "max_tokens": max_tokens,
}

stream_env = os.getenv("STREAM", "False")
# the Boolean 'stream' parameter will later be passed to Llama Stack Agents/Inference APIs
# any value non equal to 'False' will be considered as 'True'
stream = (stream_env != "False")

print(f"Inference Parameters:\n\tModel: {model_id}\n\tSampling Parameters: {sampling_params}\n\tstream: {stream}")

Connected to Llama Stack server
Inference Parameters:
	Model: granite32-8b
	Sampling Parameters: {'strategy': {'type': 'greedy'}, 'max_tokens': 512}
	stream: False


## 2. Simple Agent (Baseline)
Same agent setup as [Level 2 notebook](Level2_simple_agent_with_websearch.ipynb). 

In [5]:
agent = Agent(
    client, 
    model=model_id,
    instructions="""You are a helpful websearch assistant. When you are asked to search the latest you must use a tool. 
            Whenever a tool is called, be sure return the response in a friendly and helpful tone.
            """ ,
    tools=["builtin::websearch"],
    sampling_params=sampling_params
)
user_prompts = [
    "Are there any immediate weather-related risks in my area that could disrupt network connectivity or system availability?",
]
for prompt in user_prompts:
    print("\n"+"="*50)
    cprint(f"Processing user query: {prompt}", "blue")
    print("="*50)
    session_id = agent.create_session("web-session")
    response = agent.create_turn(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        session_id=session_id,
        stream=stream
    )
    if stream:
        for log in EventLogger().log(response):
            log.print()
    else:
        step_printer(response.steps) # print the steps of an agent's response in a formatted way. 


Processing user query: Are there any immediate weather-related risks in my area that could disrupt network connectivity or system availability?

---------- 📍 Step 1: InferenceStep ----------
🛠️ Tool call Generated:
Tool call: brave_search, Arguments: {'query': "current weather risks in user's location"}

---------- 📍 Step 2: ToolExecutionStep ----------
🔧 Executing tool...


{
│   'query': "current weather risks in user's location",
│   'top_k': [
│   │   {
│   │   │   'title': "Weather in user's location",
│   │   │   'url': 'https://www.weatherapi.com/',
│   │   │   'content': '{\'location\': {\'name\': "Kutama\'s Location", \'region\': \'Limpopo\', \'country\': \'South Africa\', \'lat\': -23.0667, \'lon\': 29.6667, \'tz_id\': \'Africa/Johannesburg\', \'localtime_epoch\': 1768824609, \'localtime\': \'2026-01-19 14:10\'}, \'current\': {\'last_updated_epoch\': 1768824000, \'last_updated\': \'2026-01-19 14:00\', \'temp_c\': 19.9, \'temp_f\': 67.8, \'is_day\': 1, \'condition\': {\'text\': \'Patchy rain nearby\', \'icon\': \'//cdn.weatherapi.com/weather/64x64/day/176.png\', \'code\': 1063}, \'wind_mph\': 14.5, \'wind_kph\': 23.4, \'wind_degree\': 107, \'wind_dir\': \'ESE\', \'pressure_mb\': 1012.0, \'pressure_in\': 29.87, \'precip_mm\': 0.05, \'precip_in\': 0.0, \'humidity\': 87, \'cloud\': 100, \'feelslike_c\': 19.9, \'feelslike_f\': 67.8, \'windchill_c\': 19.9, \'windchill_f\': 67.8, \'heatindex_c\': 19.9, \'heatindex_f\': 67.8, \'dewpoint_c\': 17.8, \'dewpoint_f\': 64.0, \'vis_km\': 10.0, \'vis_miles\': 6.0, \'uv\': 1.6, \'gust_mph\': 17.2, \'gust_kph\': 27.6}}',
│   │   │   'score': 0.9276803,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://www.severeweatheroutlook.com/2026-01-19/',
│   │   │   'title': 'Outlook for Monday, January 19 | Severe Weather Outlook',
│   │   │   'content': 'For outlooks for days 3-8, it could mean a threat of any or all of the following: wind, hail, or tornadoes. For black colored risks that means',
│   │   │   'score': 0.8124067,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://forecast.weather.gov/product.php?site=NWS&product=SRF&issuedby=TAE',
│   │   │   'title': 'Surf Zone Forecast - National Weather Service',
│   │   │   'content': '... Mon Jan 19 2026 .TODAY... Rip Current Risk............Moderate. Surf ... && Rip Current Risk Category * Low Risk - The risk for rip currents is low',
│   │   │   'score': 0.7482976,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://www.facebook.com/VoxWeather/posts/weather-maps-and-warnings-for-this-monday-19-january-2026%EF%B8%8Fimpact-based-warnings-/889804423405308/',
│   │   │   'title': 'Disruptive Rain with heavy downpours leading to localised flooding ...',
│   │   │   'content': 'WEATHER MAPS and WARNINGS for this MONDAY – 19 January 2026 ⚠️IMPACT-BASED WARNINGS issued by SAWS ⚠️ Yellow Level 2: Disruptive Rain with heavy',
│   │   │   'score': 0.7306941,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://weathershogun.com/weather/usa/fl/orlando/987/january/2026-01-19',
│   │   │   'title': 'Monday, January 19, 2026. Orlando, FL - Weather Forecast',
│   │   │   'content': 'Orlando, Florida Weather: Monday, January 19, 2026. Day 57°. Night 43°. Precipitation 7 %. Wind 6 mph. UV Index (0 - 11+) 4',
│   │   │   'score': 0.66464967,
│   │   │   'raw_content': None
│   │   }
│   ]
}


---------- 📍 Step 3: InferenceStep ----------
🤖 Model Response:
Based on the latest weather updates, there are no immediate weather-related risks in your area that could disrupt network connectivity or system availability. Here's a summary:

1. The current weather in your location (Kutama's Location, Limpopo, South Africa) is mostly cloudy with patchy rain nearby. The temperature is 19.9°C (67.8°F).

2. For the next 3-8 days, there are no severe weather warnings or alerts issued by reliable meteorological services. 

3. The rip current risk is moderate, but this doesn't directly impact network connectivity or system availability.

4. There are no reports of disruptive rain or heavy downpours that could lead to localized flooding.

5. The weather in Orlando, Florida, for comparison, is also mild with minimal precipitation and light winds.

Please note that weather conditions can change rapidly, so it's always a good idea to stay updated with local weather forecasts.

========== Query p

### Output Analysis

In this example, since the agent is unaware of the users location, it hallucinates one and generates an incorrect search query. This misidentification leads to inaccurate information about potential weather-related risks.

This is where Prompt Chaining comes in. Prompt chaining allows the agent to:
1. Maintain context across multiple queries
2. Chain multiple tools together
3. Use previous interactions to inform current decisions

Let’s see how prompt chaining can improve the accuracy of the response.

## 3. Prompt chaining with websearch tool and client tool

In this section, we demonstrate a more sophisticated use case that combines the use of two tools: location detection and web search.

1. **Automatic Location Detection**: Use the `get_location` client tool to automatically determine the user's current location.
2. **Contextual Search**: Leverage the detected location to formulate the correct websearch query.

For example, when a user asks "Are there any weather-related risks in my area that could disrupt network connectivity or system availability?", the agent will:
- First detect the user's current location using `get_location`.
- Then use that location to search for nearby weather-related risks.
- Finally, present a comprehensive response.

This demonstrates how the builtin websearch tool and custom client tools can work together to provide intelligent, context-aware responses without requiring explicit location input from the user.

In [6]:
agent = Agent(
    client, 
    model=model_id,
    instructions="""You are a helpful assistant. 
    When a user asks about their location, you MUST use the get_location tool. When you are asked to search the latest news, you MUST use the websearch tool.
    """ ,
    tools=[get_location, "builtin::websearch"],
    sampling_params=sampling_params
)
user_prompts = [
    "Where am I?",
    "Are there any immediate weather-related risks in my area that could disrupt network connectivity or system availability?"
]
session_id = agent.create_session("prompt-chaining-session")  # for prompt chaining, queries must share the same session_id.
for prompt in user_prompts:
    print("\n"+"="*50)
    cprint(f"Processing user query: {prompt}", "blue")
    print("="*50)
    response = agent.create_turn(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        session_id=session_id,
        stream=stream
    )

    if stream:
        for log in EventLogger().log(response):
            log.print()
    else:
        step_printer(response.steps) # print the steps of an agent's response in a formatted way. 


Processing user query: Where am I?

---------- 📍 Step 1: InferenceStep ----------
🛠️ Tool call Generated:
Tool call: get_location, Arguments: {'query': 'current location'}

---------- 📍 Step 2: ToolExecutionStep ----------
🔧 Executing tool...


'Your current location is: Columbus, Ohio, US'


---------- 📍 Step 3: InferenceStep ----------
🤖 Model Response:
You are currently located in Columbus, Ohio, US.

========== Query processing completed ========== 


Processing user query: Are there any immediate weather-related risks in my area that could disrupt network connectivity or system availability?

---------- 📍 Step 1: InferenceStep ----------
🛠️ Tool call Generated:
Tool call: brave_search, Arguments: {'query': 'Columbus, Ohio weather risks'}

---------- 📍 Step 2: ToolExecutionStep ----------
🔧 Executing tool...


{
│   'query': 'Columbus, Ohio weather risks',
│   'top_k': [
│   │   {
│   │   │   'title': 'Weather in Columbus, Ohio',
│   │   │   'url': 'https://www.weatherapi.com/',
│   │   │   'content': "{'location': {'name': 'Columbus', 'region': 'Ohio', 'country': 'United States of America', 'lat': 39.9611, 'lon': -82.9989, 'tz_id': 'America/New_York', 'localtime_epoch': 1768824823, 'localtime': '2026-01-19 07:13'}, 'current': {'last_updated_epoch': 1768824000, 'last_updated': '2026-01-19 07:00', 'temp_c': -4.6, 'temp_f': 23.7, 'is_day': 0, 'condition': {'text': 'Light snow', 'icon': '//cdn.weatherapi.com/weather/64x64/night/326.png', 'code': 1213}, 'wind_mph': 14.1, 'wind_kph': 22.7, 'wind_degree': 259, 'wind_dir': 'W', 'pressure_mb': 1013.0, 'pressure_in': 29.91, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 76, 'cloud': 100, 'feelslike_c': -11.5, 'feelslike_f': 11.3, 'windchill_c': -11.4, 'windchill_f': 11.4, 'heatindex_c': -5.3, 'heatindex_f': 22.4, 'dewpoint_c': -10.6, 'dewpoint_f': 13.0, 'vis_km': 8.0, 'vis_miles': 4.0, 'uv': 0.0, 'gust_mph': 19.7, 'gust_kph': 31.7}}",
│   │   │   'score': 0.8276554,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://www.dispatch.com/story/news/local/2026/01/16/columbus-zoo-closed-mlk-day-january-19-monday/88216312007/',
│   │   │   'title': 'Columbus Zoo closed Jan. 19 for safety amid winter weather',
│   │   │   'content': 'The Columbus Zoo and Aquarium will be closed Jan. 19 due to forecasted inclement weather, zoo officials announced on Facebook.',
│   │   │   'score': 0.7625837,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://www.10tv.com/article/weather/weather-blog/columbus-ohio-weather-sub-zero-wind-chills-arctic-blast-january-20-2026/530-0c377977-5477-43d2-85b5-161553ec2119',
│   │   │   'title': 'Bitter cold air moving into central Ohio to start the work week - 10TV',
│   │   │   'content': 'A 10TV Weather Impact Alert Day has been declared for Jan. 19 into the 20th, as prolonged sub-zero wind chills move into the region. #### More Videos. Example video title will go here for this video. Example video title will go here for this video. Credit: 10TV Weather Impact Team. The biggest impacts will be felt from Monday morning through Tuesday morning. During this period, temps are forecast to be as cold as 4 degrees early Tuesday, with wind chills pushing down between -5 and -15 degrees starting as soon as Monday evening. Due to this dangerously cold air, we have issued a 10TV Weather Impact Alert Day starting Monday morning and continuing through Tuesday morning. Wind chills will first start to go sub-zero in our northwestern counties toward midday Monday, then progress across the 10TV viewing areas as temps fall through Monday afternoon. Credit: 10TV Weather Impact Team. Credit: 10TV Weather Impact Team. ***--The 10 Weather Impact Team***.',
│   │   │   'score': 0.61779,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://www.youtube.com/watch?v=ZLu4FUJ4mLE',
│   │   │   'title': 'Columbus, Ohio weather forecast for Jan. 15, 2026 - YouTube',
│   │   │   'content': "Columbus, Ohio weather forecast for Jan. 15, 2026 | On/off flurries and much colder\nWBNS 10TV\n203000 subscribers\n48 likes\n2743 views\n15 Jan 2026\nIsolated heavy snow bands remain possible through the overnight, mostly south and west of Columbus. There is a Winter Weather Advisory that has been issued through 9 AM that extends from Franklin County to the south and southwest. Although most locations could see minimal accumulations of an inch or less, with isolated heavy snow bands, visibility could drop quickly and we could pick up an inch or two.\r\n\r\nWith the timing during the commute, this will lead to some delays on the roads and has prompted some snow emergencies and a handful of 2-hour delays for schools this morning. Please plan for extra time with your commute as there could also be some slick and even icy spots as temper


---------- 📍 Step 3: InferenceStep ----------
🤖 Model Response:
According to the latest weather forecasts, Columbus, Ohio is expected to experience prolonged sub-zero wind chills starting from Monday evening, with temperatures as cold as 4 degrees early Tuesday. Wind chills could push down between -5 and -15 degrees. The Columbus Zoo and Aquarium has also announced its closure on January 19 due to the forecasted inclement weather. While there are no immediate weather-related risks that could disrupt network connectivity or system availability, the extreme cold could potentially cause some disruptions, especially with regards to outdoor activities and transportation. It's recommended to stay indoors, bundle up, and drive safely.

========== Query processing completed ========== 



### ReAct Agent with websearch tool and client tool

This section demonstrates the ReAct (Reasoning and Acting) framework in action.

Here is a walkthrough of how the ReAct agent will tackle this same "weather near me" problem:

When asked "Are there any weather-related risks in my area that could disrupt network connectivity or system availability?", the agent will:

1. **Reason** that it needs to get location information first.
2. **Act** by calling the `get_location` client tool.
3. **Observe** the location result.
4. **Reason** that it now needs to search for weather in that location.
5. **Act** by calling the `websearch` tool with observed location.
6. **Observe** and processes the search results into a final answer. 

Unlike prompt chaining which follows fixed steps, ReAct dynamically breaks down tasks and adapts its approach based on the results of each step. This makes it more flexible and capable of handling complex, real-world queries effectively.

In [7]:
agent = ReActAgent(
            client=client,
            model=model_id,
            tools=[get_location, "builtin::websearch"],
            response_format={
                "type": "json_schema",
                "json_schema": ReActOutput.model_json_schema(),
            },
            sampling_params=sampling_params,
        )
user_prompts = [
    "Are there any immediate weather-related risks in my area that could disrupt network connectivity or system availability?"
]
session_id = agent.create_session("web-session")
for prompt in user_prompts:
    print("\n"+"="*50)
    cprint(f"Processing user query: {prompt}", "blue")
    print("="*50)
    response = agent.create_turn(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        session_id=session_id,
        stream=stream
    )
    if stream:
        for log in EventLogger().log(response):
            log.print()
    else:
        step_printer(response.steps) # print the steps of an agent's response in a formatted way. 


Processing user query: Are there any immediate weather-related risks in my area that could disrupt network connectivity or system availability?

---------- 📍 Step 1: InferenceStep ----------
🤖 Model Response:
{
    "thought": "I need to get the user's current location and then check for any weather-related risks in that area. I will use the `get_location` tool to find the user's location and then the `web_search` tool to check for any weather-related risks.",
    "action": {
        "tool_name": "get_location",
        "tool_params": [{"name": "query", "value": "user's current location"}]
    },
    "answer": null
}


---------- 📍 Step 2: ToolExecutionStep ----------
🔧 Executing tool...


'Your current location is: Columbus, Ohio, US'


---------- 📍 Step 3: InferenceStep ----------
🤖 Model Response:
{
    "thought": "Now that I have the user's location, I will use the `web_search` tool to check for any weather-related risks in Columbus, Ohio.",
    "action": {
        "tool_name": "web_search",
        "tool_params": [{"name": "query", "value": "weather-related risks in Columbus, Ohio"}]
    },
    "answer": null
}


---------- 📍 Step 4: ToolExecutionStep ----------
🔧 Executing tool...


{
│   'query': 'weather-related risks in Columbus, Ohio',
│   'top_k': [
│   │   {
│   │   │   'title': 'Weather in Columbus, Ohio',
│   │   │   'url': 'https://www.weatherapi.com/',
│   │   │   'content': "{'location': {'name': 'Columbus', 'region': 'Ohio', 'country': 'United States of America', 'lat': 39.9611, 'lon': -82.9989, 'tz_id': 'America/New_York', 'localtime_epoch': 1768824823, 'localtime': '2026-01-19 07:13'}, 'current': {'last_updated_epoch': 1768824000, 'last_updated': '2026-01-19 07:00', 'temp_c': -4.6, 'temp_f': 23.7, 'is_day': 0, 'condition': {'text': 'Light snow', 'icon': '//cdn.weatherapi.com/weather/64x64/night/326.png', 'code': 1213}, 'wind_mph': 14.1, 'wind_kph': 22.7, 'wind_degree': 259, 'wind_dir': 'W', 'pressure_mb': 1013.0, 'pressure_in': 29.91, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 76, 'cloud': 100, 'feelslike_c': -11.5, 'feelslike_f': 11.3, 'windchill_c': -11.4, 'windchill_f': 11.4, 'heatindex_c': -5.3, 'heatindex_f': 22.4, 'dewpoint_c': -10.6, 'dewpoint_f': 13.0, 'vis_km': 8.0, 'vis_miles': 4.0, 'uv': 0.0, 'gust_mph': 19.7, 'gust_kph': 31.7}}",
│   │   │   'score': 0.8375485,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://weathershogun.com/weather/usa/oh/columbus/3538/january/2026-01-19',
│   │   │   'title': 'Monday, January 19, 2026. Columbus, OH - Weather Forecast',
│   │   │   'content': 'Columbus, Ohio Weather: Monday, January 19, 2026. Day 30°. Night 19°. Precipitation 25 %. Wind 7 mph. UV Index (0 - 11+) 1',
│   │   │   'score': 0.8091708,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://www.wunderground.com/severe/us/oh/columbus',
│   │   │   'title': 'Columbus, OH Severe Weather Alert - Weather Underground',
│   │   │   'content': 'Active Weather Alerts. Cold Weather Advisory. Issued: 12:22 AM Jan. 19, 2026 – National Weather Service ...COLD WEATHER ADVISORY REMAINS IN EFFECT FROM 7 AM',
│   │   │   'score': 0.7197191,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://www.wunderground.com/severe/us/oh/columbus/43201',
│   │   │   'title': 'Columbus, OH Severe Weather Alert | Weather Underground',
│   │   │   'content': 'A few hours of snow showers and west wind gusts of 35 to 40 mph will occur with the frontal passage. While any accumulations will generally be a half inch or',
│   │   │   'score': 0.5510187,
│   │   │   'raw_content': None
│   │   },
│   │   {
│   │   │   'url': 'https://www.nbc4i.com/video/as-temperatures-fall-library-opens-zoo-closes/11438801/',
│   │   │   'title': 'As temperatures fall, library opens, zoo closes – NBC4 WCMH-TV',
│   │   │   'content': 'Storm Team 4 forecast with Bryan Still. Weather / Jan 19, 2026 / 05 ... Latest; Local News; U.S./World; Buckeyes; Daytime Columbus. Latest',
│   │   │   'score': 0.4785538,
│   │   │   'raw_content': None
│   │   }
│   ]
}


---------- 📍 Step 5: InferenceStep ----------
🤖 Model Response:
{
    "thought": "Based on the search results, there is a Cold Weather Advisory in effect from 7 AM on January 19, 2026, in Columbus, Ohio. This could potentially disrupt network connectivity or system availability due to the severe weather conditions.",
    "action": null,
    "answer": "There is a Cold Weather Advisory in effect from 7 AM on January 19, 2026, in Columbus, Ohio. This could potentially disrupt network connectivity or system availability due to the severe weather conditions."
}

========== Query processing completed ========== 



## Key Takeaways
- This notebook demonstrated how to build more capable agents using Prompt Chaining and the ReAct framework.
- It showed how agents can maintain context across multiple steps and perform structured, multi-step reasoning.
- It highlights how ReAct enables dynamic tool selection and adaptive decision-making based on intermediate results.
- These techniques enhance agent autonomy and make them more suitable for complex operational tasks.

For further extensions, continue exploring in the next notebook: [RAG Agents](Level4_RAG_agent.ipynb).

#### Any Feedback?

If you have any feedback on this or any other notebook in this demo series we'd love to hear it! Please go to https://www.feedback.redhat.com/jfe/form/SV_8pQsoy0U9Ccqsvk and help us improve our demos. 